# Pipeline RAG — Retrieval Augmented Generation

## Comment ça fonctionne ?

Un pipeline RAG se déroule en **3 étapes** à chaque question :

```
Question utilisateur
      ↓
[1. RETRIEVAL] — on transforme la question en vecteur (embedding),
                 on cherche les N événements les plus proches dans FAISS
      ↓
[2. AUGMENTATION] — on injecte ces événements dans un prompt contextualisé
      ↓
[3. GENERATION] — on envoie le prompt à Mistral qui génère la réponse finale
```

**Pourquoi ne pas juste envoyer la question à Mistral directement ?**  
Mistral ne connaît pas les événements de Rennes — ses données d'entraînement sont figées dans le temps.  
Le RAG lui donne accès à nos données en temps réel, sans le ré-entraîner.

## Ce qu'on va construire

1. Charger l'index FAISS et le mapping
2. Tester la recherche de documents similaires (retrieval)
3. Construire le prompt augmenté
4. Générer une réponse avec Mistral
5. Assembler le pipeline complet

In [1]:
import os
import faiss
import numpy as np
import pandas as pd
from langchain_mistralai import MistralAIEmbeddings

# Chargement de l'index FAISS et du mapping position → corpus
index = faiss.read_index("../data/faiss_index/events.index")
mapping = pd.read_csv("../data/faiss_index/events_mapping.csv", index_col=0)

print(f"Index chargé : {index.ntotal} vecteurs de dimension {index.d}")
print(f"Mapping chargé : {len(mapping)} lignes")

Index chargé : 5030 vecteurs de dimension 1024
Mapping chargé : 5030 lignes


## 1. Retrieval — recherche des documents pertinents

On transforme la question en vecteur avec le **même modèle d'embedding** utilisé lors de l'indexation (`mistral-embed`).  
C'est crucial : si on embeddait avec un modèle différent, les vecteurs ne seraient pas comparables.

FAISS calcule ensuite la distance euclidienne entre ce vecteur et tous les vecteurs de l'index,  
et retourne les `k` plus proches — ce sont les événements les plus "sémantiquement proches" de la question.

In [2]:
embed_model = MistralAIEmbeddings(
    model="mistral-embed",
    mistral_api_key=os.getenv("MISTRAL_API_KEY")
)

question = "Quels concerts sont prévus à Rennes ?"

# On embede la question et on la convertit en tableau numpy float32
question_vector = np.array([embed_model.embed_query(question)], dtype=np.float32)

# k = nombre de documents à récupérer
k = 5
distances, indices = index.search(question_vector, k)

print(f"Top {k} résultats pour : '{question}'\n")
for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    titre = mapping.iloc[idx]["corpus"].split("\n")[0]
    print(f"{rank+1}. [distance: {dist:.2f}] {titre}")

Top 5 résultats pour : 'Quels concerts sont prévus à Rennes ?'

1. [distance: 0.39] Titre : Le concert du mardi - 𝗘́𝗹𝗶𝘀𝗮 𝗰𝗵𝗮𝗻𝘁𝗲 𝗝𝗲𝗮𝗻𝗻𝗲
2. [distance: 0.41] Titre : Concert Jelias
3. [distance: 0.41] Titre : Le Grand Concert de Noël
4. [distance: 0.41] Titre : Le concert du mardi - 𝗔𝗿𝗼̂𝗺𝗲(𝘀)
5. [distance: 0.42] Titre : KEREN ANN EN CONCERT


## 2. Augmentation — construction du prompt

On injecte les documents récupérés dans un prompt structuré.  
Le prompt dit à Mistral : "voici des informations sur des événements, réponds à la question en te basant uniquement sur ces informations".

Cette contrainte ("uniquement sur ces informations") est importante pour la **faithfulness** — une des métriques RAGAS qu'on évaluera plus tard. Elle évite que Mistral invente des événements qui n'existent pas.

In [3]:
# Récupération des corpus des k événements trouvés
context_docs = [mapping.iloc[idx]["corpus"] for idx in indices[0]]

# Assemblage du contexte : chaque document séparé par une ligne
context = "\n\n---\n\n".join(context_docs)

prompt = f"""Tu es un assistant spécialisé dans les événements culturels à Rennes.
Réponds à la question en te basant uniquement sur les événements fournis ci-dessous.
Si l'information n'est pas dans le contexte, dis-le clairement.

ÉVÉNEMENTS :
{context}

QUESTION : {question}

RÉPONSE :"""

print(prompt)

Tu es un assistant spécialisé dans les événements culturels à Rennes.
Réponds à la question en te basant uniquement sur les événements fournis ci-dessous.
Si l'information n'est pas dans le contexte, dis-le clairement.

ÉVÉNEMENTS :
Titre : Le concert du mardi - 𝗘́𝗹𝗶𝘀𝗮 𝗰𝗵𝗮𝗻𝘁𝗲 𝗝𝗲𝗮𝗻𝗻𝗲
Description : Élisa Chante Jeanne “Élisa Chante Jeanne” est un quartet swing en hommage à Jeanne Moreau. Des chansons écrites par Serge Rezvani, alias Cyrus Bassiak, le poète Norge et Jeanne Moreau elle-même. C'est avec beaucoup d'émotion que vous baladeront les chansons écrites par ou pour Jeanne et chantées par Élisa. Vous connaissez sans doute les célèbres Le Tourbillon De La Vie et J’ai La Mémoire Qui Flanche , qui sauront vous faire chanter en chœur. Vous découvrirez aussi des pépites cachées de la chanson française. Distribution : Elisa Robin - Chant Olivier Roth - Guitare Bérenger Heurtaux - Guitare Lyn Aubert - Contrebasse 1er set : 18h devant la Maison du Parc (av. André Malraux, Rennes) 2e set : 1

## 3. Génération — réponse de Mistral

On envoie le prompt augmenté à Mistral qui génère la réponse finale.  
On utilise `mistral-small-latest` — un bon compromis qualité/coût pour la génération de texte.

In [4]:
from mistralai.client import MistralClient
from mistralai.models.chat_completion import ChatMessage

client = MistralClient(api_key=os.getenv("MISTRAL_API_KEY"))

response = client.chat(
    model="mistral-small-latest",
    messages=[ChatMessage(role="user", content=prompt)]
)

print(response.choices[0].message.content)

Voici les concerts prévus à Rennes selon les événements fournis :

1. **Le concert du mardi - Élisa chante Jeanne**
   - **Date** : 22 juillet 2025
   - **Horaire** : 18h (1er set) et 19h (2e set)
   - **Lieu** : Devant la Maison du Parc (av. André Malraux) et place Eugène Aulnette
   - **Tarif** : Entrée libre

2. **Concert Jelias**
   - **Date** : 7 novembre 2025
   - **Horaire** : 17h30 - 18h15
   - **Lieu** : Hors les murs, Les Champs Libres (10 Cours des Alliés)
   - **Tarif** : Gratuit

3. **Le Grand Concert de Noël**
   - **Date** : 14 décembre 2025
   - **Horaire** : 19h
   - **Lieu** : Salle polyvalente, Association du Bourg L'Evêque (16 rue Papu)
   - **Tarif** : Entrée libre

4. **Le concert du mardi - Arôme(s)**
   - **Date** : 8 juillet 2025
   - **Horaire** : 16h - 17h30
   - **Lieu** : Pump track du parc de Quincé (rue Aurélie Nemours)
   - **Tarif** : Entrée libre

5. **Keren Ann en concert**
   - **Date** : 7 mai 2026
   - **Horaire** : 18h30 - 20h
   - **Lieu** : Théâ

## 4. Pipeline complet — fonction `ask()`

On regroupe les 3 étapes en une seule fonction réutilisable.  
C'est cette fonction qui sera importée par l'API FastAPI.

In [5]:
def ask(question: str, k: int = 5) -> str:
    """Pipeline RAG complet : retrieval → augmentation → génération."""
    # 1. Retrieval
    question_vector = np.array([embed_model.embed_query(question)], dtype=np.float32)
    _, indices = index.search(question_vector, k)
    context_docs = [mapping.iloc[idx]["corpus"] for idx in indices[0]]
    context = "\n\n---\n\n".join(context_docs)

    # 2. Augmentation
    prompt = f"""Tu es un assistant spécialisé dans les événements culturels à Rennes.
Réponds à la question en te basant uniquement sur les événements fournis ci-dessous.
Si l'information n'est pas dans le contexte, dis-le clairement.

ÉVÉNEMENTS :
{context}

QUESTION : {question}

RÉPONSE :"""

    # 3. Génération
    response = client.chat(
        model="mistral-small-latest",
        messages=[ChatMessage(role="user", content=prompt)]
    )
    return response.choices[0].message.content


# Test avec une question différente
print(ask("Y a-t-il des expositions gratuites à Rennes ?"))

Oui, il y a des expositions gratuites à Rennes ! Les musées et centres d’art de la Ville de Rennes proposent des **collections et expositions permanentes gratuites** tous les jours, selon les horaires suivants :

- **Musée des beaux-arts** (quai Émile Zola) : 10h–18h
- **Musée des beaux-arts Maurepas** (2 allée Georges-de-la-Tour) : 14h–19h
- **Musée de Bretagne** (aux Champs Libres) : 14h–19h
- **Écomusée de la Bintinais** : 14h–19h
- **La Criée, centre d’art contemporain** (place Honoré Commeurec) : 14h–19h
- **FRAC Bretagne** (19 avenue André Mussat) : 12h–19h

De plus, **les expositions temporaires sont gratuites le premier dimanche du mois** dans certains musées (renseignez-vous sur leurs sites respectifs pour confirmation).

*Source : Événements "Musées gratuits!" (Rennes Métropole).*


## 5. Test hors contexte

On vérifie que le modèle respecte la contrainte du prompt et ne répond pas en dehors des événements fournis.

In [6]:
print(ask("Quelle est la capitale de l'Allemagne ?"))

Cette information ne figure pas dans les événements culturels fournis. Je ne peux donc pas répondre à votre question.
